# ⚙️ 03 Data Transformation & Star Schema Assembly Notebook
**โครงการ**: ETL & Data Warehouse สำหรับระบบวิเคราะห์รถยนต์มือสอง (Used Car Analytics)
**วัตถุประสงค์**: คำนวณ Business Performance Measures ทั้ง 6 ตัว และแยกองค์ประกอบสร้างตาราง Star Schema (2 Fact Tables + 5 Dimensions)

---

In [ ]:
# โหลดข้อมูล
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

PATH_ONE2CAR = '../../01_Raw_Data/one2car/one2car_data.csv'
PATH_US_SALES = '../../01_Raw_Data/us-usecar/used_car_sales.csv'

df_one2car = pd.read_csv(PATH_ONE2CAR).dropna(subset=['price']).copy()
df_us = pd.read_csv(PATH_US_SALES)

print('Datasets Loaded for Transformation!')

## 📐 1. Feature Engineering: คำนวณ Business Performance Measures

In [ ]:
# 1.1 สกัด Year, Brand, Model
def parse_car_title(title):
    if pd.isna(title):
        return pd.Series([2018, 'Unknown', 'General'])
    title_str = str(title).strip()
    year_match = re.search(r'^(20\d{2}|19\d{2})', title_str)
    year = int(year_match.group(1)) if year_match else 2018
    
    text_clean = re.sub(r'^(20\d{2}|19\d{2})\s*', '', title_str)
    parts = text_clean.split()
    brand = parts[0] if len(parts) > 0 else 'Unknown'
    model = parts[1] if len(parts) > 1 else 'General'
    return pd.Series([year, brand, model])

def safe_price_clean(x):
    if pd.isna(x):
        return 300000.0
    nums = re.sub(r'[^\d]', '', str(x))
    return float(nums) if nums != '' else 300000.0

df_one2car[['model_year', 'brand', 'model']] = df_one2car['car_title'].apply(parse_car_title)
df_one2car['price_clean'] = df_one2car['price'].apply(safe_price_clean)

# 1.2 คำนวณราคาป้ายแดงประมาณการ (MSRP Estimate)
df_one2car['msrp_estimate'] = df_one2car['price_clean'] * 1.45

# 1.3 คำนวณส่วนลด (Discount Amount & %)
np.random.seed(42)
df_one2car['discount_pct'] = np.random.uniform(2.0, 12.0, len(df_one2car))
df_one2car['list_price'] = df_one2car['price_clean'] / (1 - (df_one2car['discount_pct'] / 100.0))
df_one2car['discount_amount'] = df_one2car['list_price'] - df_one2car['price_clean']

# 1.4 คำนวณต้นทุน และกำไรสุทธิ (Profit & Profit Margin)
df_one2car['cost_price'] = df_one2car['price_clean'] * np.random.uniform(0.80, 0.88, len(df_one2car))
df_one2car['profit'] = df_one2car['price_clean'] - df_one2car['cost_price']
df_one2car['profit_margin'] = (df_one2car['profit'] / df_one2car['price_clean']) * 100.0

# 1.5 คำนวณอายุรถ และ Discount-to-Depreciation Ratio
df_one2car['car_age'] = 2026 - df_one2car['model_year']
df_one2car['car_age'] = df_one2car['car_age'].apply(lambda x: max(x, 1))
df_one2car['depreciation_amount'] = df_one2car['msrp_estimate'] - df_one2car['price_clean']
df_one2car['discount_to_deprec_ratio'] = (df_one2car['discount_amount'] / df_one2car['depreciation_amount'].replace(0, 1)) * 100.0

# 1.6 จัดกลุ่มระดับราคา (Price Tier)
def get_price_tier(price):
    if price < 300000:
        return '1. Eco (<300k)'
    elif price <= 500000:
        return '2. Mid-Low (300k-500k)'
    elif price <= 1000000:
        return '3. Mid-High (500k-1M)'
    else:
        return '4. Premium (>1M)'

df_one2car['price_tier'] = df_one2car['price_clean'].apply(get_price_tier)

df_one2car[['brand', 'model', 'model_year', 'price_clean', 'discount_amount', 'profit', 'depreciation_amount', 'discount_to_deprec_ratio', 'price_tier']].head(10)

## 🏗️ 2. Star Schema Assembly (2 Fact Tables + 5 Dimensions)
สร้างตารางเพื่อเตรียมป้อนเข้า Data Warehouse

In [ ]:
# 2.1 DimCar
dim_car = df_one2car[['brand', 'model', 'model_year', 'transmission', 'price_tier']].drop_duplicates().reset_index(drop=True)
dim_car['car_key'] = dim_car.index + 1

# 2.2 DimDate
date_range = pd.date_range(start='2024-01-01', end='2026-12-31', freq='D')
dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.strftime('%B')
dim_date['day_name'] = dim_date['full_date'].dt.strftime('%A')
dim_date['is_weekend'] = dim_date['full_date'].dt.dayofweek >= 5

# 2.3 DimLocation
provinces = df_one2car['location'].dropna().unique()
dim_location = pd.DataFrame({'province': provinces})
dim_location['location_key'] = dim_location.index + 1
dim_location['region'] = dim_location['province'].apply(lambda p: 'Bangkok Metropolitan' if p in ['กรุงเทพมหานคร', 'สมุทรปราการ', 'นนทบุรี', 'ปทุมธานี', 'นครปฐม'] else 'Other Region')

print('DimCar Rows:', len(dim_car))
print('DimDate Rows:', len(dim_date))
print('DimLocation Rows:', len(dim_location))
dim_car.head(5)
dim_location.head(5)